# POC lengkap — Mac Apple Silicon, RAM 24 GB

Konfigurasi sudah disiapkan: **ACE-Step Base · MPS · bass · seed 0 · CPU offload**.
JupyterLab boleh tetap menggunakan Python 3.15; Python model 3.11 terpisah.
Dependency pada lockfile upstream memerlukan macOS 14 atau lebih baru.

**Sebelum Run All:** letakkan satu rekaman gitar/piano (minimal 10 detik) di
folder `data/` repository melalui file browser JupyterLab. Folder dibuat oleh sel
pertama jika belum ada. Bisa WAV/MP3/M4A; jika ada beberapa file, gunakan nama
`input.wav` untuk WAV yang dipilih atau isi SOURCE pada sel konfigurasi.

**Run All akan memasang dependency ML, mengunduh bobot sekitar 14.88 GB,
memverifikasi checksum, dan mencoba generasi nyata pada Mac-mu.** Tidak perlu
mengubah INSTALL_RUNTIME/DOWNLOAD_WEIGHTS/GENERATE. Tidak ada fixture palsu atau
klaim sukses sebelum inferensi. Unduhan pertama dan kompilasi awal dapat lama;
sel yang sedang berjalan bertanda `[*]`. Simpan traceback jika gagal.

Sel-sel berjalan berurutan dan berhenti jika prasyarat gagal. Tidak ada gerbang
lisensi untuk POC. Data, bobot, cache dan hasil tetap lokal, tidak dipush otomatis.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, subprocess, sys, uuid
from IPython.display import Audio, display

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "configs/mac-24gb.json").is_file(), "Buka notebook dari repository StemsGenerating-Models"
sys.path.insert(0, str(ROOT / "scripts"))
sys.path.insert(0, str(ROOT / "tools/stem_eval"))
from diagnose import diagnose
from mac_workflow import choose_source, check_model_runtime, run_command
from setup_runtime import runtime_python, UPSTREAM
from system_tools import require_audio_tools

DATA = ROOT / "data"
REPORTS = ROOT / "reports"
DATA.mkdir(exist_ok=True)
REPORTS.mkdir(exist_ok=True)
HARDWARE = diagnose(ROOT)
(REPORTS / "hardware.json").write_text(json.dumps(HARDWARE, indent=2))
print(json.dumps(HARDWARE, indent=2))


## 1. Rekaman dan konfigurasi sudah siap

`SOURCE=None` otomatis memilih satu file di data/, atau input.wav jika tersedia.
Satu-satunya hal yang perlu kamu sediakan adalah rekaman sumber. Default memakai
10 detik pertama. Audio biasa memakai conditioning audio-only; untuk ekspor
SongMap, isi CONDITIONING dan gunakan START=0 karena WAV sudah dipotong.

`DRY_RUN=False` adalah mode lengkap. True hanya untuk pengujian pipeline tanpa
instalasi/model; hasil dry run tidak boleh disebut hasil generasi.


In [ ]:
SOURCE = None
CONDITIONING = None
DRY_RUN = False

PROFILE = json.loads((ROOT / "configs/mac-24gb.json").read_text())
START = PROFILE["start_seconds"]
DURATION = PROFILE["duration_seconds"]
require_audio_tools()
if not DRY_RUN and not HARDWARE["apple_silicon"]:
    raise RuntimeError("Notebook ini disiapkan untuk Mac Apple Silicon. Untuk CUDA gunakan notebook 01.")
source = choose_source(DATA, SOURCE)
ASSETS = ROOT / "configs/assets-base.json"
assets = json.loads(ASSETS.read_text())
print("Sumber:", source)
print("Profil:", json.dumps(PROFILE, indent=2))
print("Unduhan bobot pertama:", round(assets["download_bytes"] / 1e9, 2), "GB; belum termasuk dependency runtime")
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:6]
WORK = ROOT / "runs" / ("mac24_" + RUN_ID)
WORK.mkdir(parents=True)
(WORK / "profile.json").write_text(json.dumps(PROFILE, indent=2))


## 2. Potong, dengarkan, dan catat rencana

Gain sumber tetap 0.7, stereo PCM16 32 kHz. Tidak ada beat/chord buatan.
Rencana harus menunjukkan satu job, seed 0, offload=true dan device=mps.


In [ ]:
from local_clip import local_clip
PREPARED = WORK / "prepared"
local_clip(source, PREPARED, START, DURATION, Path(CONDITIONING) if CONDITIONING else None)
display(Audio(filename=str(PREPARED / "source.wav")))
PLAN = WORK / "plan"
subprocess.run(run_command(ROOT, PROFILE, PREPARED, PLAN), check=True)
print((PLAN / "experiment.json").read_text())


## 3. Instal dependency model otomatis, lalu periksa MPS

Checkout source dipin; `uv sync --frozen --python 3.11` memakai environment
vendor/ACE-Step-1.5/.venv. Notebook tidak memasang PyTorch ke kernel Jupyter 3.15.
Sel ini dapat dijalankan lagi untuk menyegarkan dependency sesuai lockfile.
Unduhan bobot hanya dilanjutkan setelah Python 3.11 dan MPS terverifikasi.


In [ ]:
if DRY_RUN:
    print("DRY RUN: instalasi dan pemeriksaan runtime model tidak dijalankan")
else:
    subprocess.run([sys.executable, str(ROOT / "scripts/setup_runtime.py"), "--install"], check=True)
    runtime_report = check_model_runtime(runtime_python())
    (WORK / "runtime.json").write_text(json.dumps(runtime_report, indent=2))
    print(json.dumps(runtime_report, indent=2))


## 4. Unduh dan verifikasi bobot otomatis

Butuh sekitar 32.7 GiB headroom untuk aset lengkap yang belum ada, ditambah ruang
runtime. Revisi/checksum dipin pada manifest. File yang sudah ada tetap diverifikasi;
tidak mengubah checksum yang diharapkan jika unduhan rusak.
Initializer upstream meminta komponen utama termasuk Turbo/LM, walaupun run ini
memakai Base dan planner mati. Bobot tidak masuk Git.


In [ ]:
if DRY_RUN:
    print("DRY RUN: tidak mengunduh atau memverifikasi bobot model")
else:
    subprocess.run([str(runtime_python()), str(ROOT / "scripts/model_assets.py"),
                    "--manifest", str(ASSETS), "--upstream", str(UPSTREAM), "--download"], check=True)


## 5. Generate satu stem bass

Base lego 50 langkah, CFG 7, planner/CoT mati. MPS + CPU offload. Selama notebook
menampilkan [*], proses masih berjalan; belum ada estimasi waktu untuk Mac ini.
24 GB RAM tidak menjamin bebas OOM. Berhenti dan kirim error jika gagal; jangan
menganggap penambahan disk menyelesaikan kekurangan memori terpadu.


In [ ]:
RESULTS = WORK / "generated"
if DRY_RUN:
    print("DRY RUN: inferensi tidak dijalankan; tidak ada stem model")
else:
    subprocess.run(run_command(ROOT, PROFILE, PREPARED, RESULTS, execute=True), check=True)


## 6. Dengarkan stem dan mix, lalu nilai kualitas

Mix = 0.5 × sumber bersih + 0.5 × stem. Dengarkan harmoni, groove, jeda, dinamika,
serta kebocoran instrumen sumber. MPS current memory bukan peak memori total dan
keberhasilan Mac bukan bukti Core AI/iPhone. Hasil tersimpan di folder WORK.


In [ ]:
from summarize import summarize
if RESULTS.exists() and list(RESULTS.glob("*/result.json")):
    reports = summarize(RESULTS)
    print((RESULTS / "summary.csv").read_text())
    for file in sorted(RESULTS.glob("*/stem.wav")):
        print("STEM:", file.parent.name)
        display(Audio(filename=str(file)))
    for file in sorted(RESULTS.glob("*/mix.wav")):
        print("MIX:", file.parent.name)
        display(Audio(filename=str(file)))
else:
    print("Belum ada hasil inferensi nyata.")
print("Folder run:", WORK)


## 7. Percobaan berikutnya

Jika bass pertama sudah berhasil: ubah instrument di configs/mac-24gb.json menjadi
`drums`, atau tambah seed menjadi [0,1,2] untuk perbandingan. Notebook selalu
membuat folder run baru dan tidak menimpa hasil lama. Dependency dan aset
terverifikasi dipakai kembali; menutup kernel tidak menghapusnya.

Untuk feedback, kirim hardware report, traceback, atau summary.csv yang sudah
ditinjau. Audio dan notebook dengan output audio jangan dimasukkan Git.
Notebook ini belum diuji dengan inferensi Mac nyata oleh agen; itu dilakukan di
mesin lokalmu. Tidak ada model baru yang dilatih pada langkah ini.
